# 06 · RAG v2: Widen the corpus and write fresh evaluation queries


**Why.** The English advisory can only use English passages, but the FEMA P-2406 passages (the ones most
relevant to damaged buildings) exist in the corpus **only in Arabic**. So the English advisory never sees them.
Passages are also bare one-sentence snippets, which makes dense retrieval noisy.

**What this notebook does**
1. Lists the Arabic-only FEMA passages and writes a template for their English originals.
2. (Optional) fills missing English text with a machine translation, clearly marked as such.
3. Checks the pairs are aligned (an Arabic passage and its English twin should embed close together).
4. Adds a *context prefix* (source and section) to each passage, used **only for embedding**, never shown.
5. Writes the new corpus `passages_v3.csv`.
6. Writes 20 **fresh** evaluation queries (`fresh_queries.json`) *before* any retrieval is changed, so the
   old queries, which were used to tune the 0.70 cutoff, do not flatter the baseline.

**Outputs** (all in `DRIVE_DIR`): `fema_en_pairs_template.csv` (you fill), `passages_v3.csv`, `fresh_queries.json`.
Run notebook 07 afterwards.

In [ ]:
!pip -q install sentence-transformers
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# ---- CONFIG: edit these to match your files on the shared Drive -----------------------
DRIVE_DIR = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT/guidance"   # folder holding the csv files
PASSAGES_CSV = f"{DRIVE_DIR}/passages_v2.csv"   # columns: id, pair_id, source, language, licence_note, text
OLD_SHEET    = f"{DRIVE_DIR}/eval_sheet_v2_labelled.csv"     # the old 20-query labelled sheet
PAIRS_CSV    = f"{DRIVE_DIR}/fema_en_pairs.csv"              # upload the pre-filled file here

FEMA_ID_PREFIX = "FEMA-"      # Arabic-only passages have ids FEMA-G01-ar, FEMA-G02-ar, ...
# Source line given to the English twins: the English original of the Arabic sheet (P-2406 = V-1005 in Arabic).
EN_SOURCE = "FEMA V-1005 (July 2024), Be Prepared for a Flood, English original of P-2406, ready.gov"

## 1. Load and inspect

Always print the columns first. If a name in the config does not match, fix it there rather than in the code below.

In [ ]:
passages = pd.read_csv(PASSAGES_CSV, encoding="utf-8-sig").to_dict("records")
print("passage columns:", list(passages[0].keys()))

def corpus_overview(passages):
    """Counts of passages per (source, language). The FEMA row has Arabic only, so English
    queries can never retrieve it (the advisory filters passages by its own language)."""
    df = pd.DataFrame(passages)
    return df.groupby(["source", "language"]).size().unstack(fill_value=0)

corpus_overview(passages)

## 2. Arabic-only FEMA passages and the English pairs file

`list_ar_only` finds Arabic passages (`FEMA-Gxx-ar`) that have no English twin (`FEMA-Gxx-en`) yet.
`fema_en_pairs.csv` has the columns `ar_id`, `ar_text`, `en_text`, `translation`. **It comes pre-filled** with the matching sentences from the
English original, FEMA V-1005 (July 2024), checked sentence by sentence against the Arabic. Upload it to `DRIVE_DIR`.
The `translation` column records how each English text was produced (`human` or `machine`); we report that in the paper.

In [ ]:
def twin_id(ar_id):
    """FEMA-G08-ar -> FEMA-G08-en (the same convention as the NCEMA and READY passages)."""
    return ar_id[:-3] + "-en"

def list_ar_only(passages, id_prefix):
    """Arabic passages whose id starts with id_prefix and that have no English twin yet."""
    have = {p["id"] for p in passages}
    return [p for p in passages
            if p["language"] == "ar" and p["id"].startswith(id_prefix) and twin_id(p["id"]) not in have]

def write_pairs_template(rows, path):
    """Write the file you will fill in. Does not overwrite an existing, partly filled file."""
    import os
    if os.path.exists(path):
        print("Pairs file already exists, leaving it alone:", path); return
    pd.DataFrame({"ar_id": [r["id"] for r in rows], "ar_text": [r["text"] for r in rows],
                  "en_text": "", "translation": "human"}).to_csv(path, index=False, encoding="utf-8-sig")
    print(f"wrote template with {len(rows)} rows ->", path)

todo = list_ar_only(passages, FEMA_ID_PREFIX)
print(len(todo), "Arabic-only FEMA passages")
write_pairs_template(todo, PAIRS_CSV)     # does nothing if you already uploaded the pre-filled file

**The pairs file is already filled** (English sentences from FEMA V-1005). Skim it once against the Arabic, and edit any row you disagree with.
FEMA publications are generally US-government public domain; `licence_note` in the passages file carries the licence text.

**Fallback only:** the optional cell below fills *empty* `en_text` cells with a Qwen3-4B translation and marks those rows `machine`.
You should not need it for the FEMA sheet.

In [ ]:
def machine_translate(texts, model_name="Qwen/Qwen3-4B", max_new_tokens=160):
    """Greedy Arabic->English translation with a small instruct model. Fallback only."""
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_name)
    lm = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
    out = []
    for t in texts:
        msg = [{"role": "user", "content":
                "Translate this Arabic flood-safety guidance into plain English. "
                "Output only the translation and add nothing.\n\n" + t}]
        prompt = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        ids = tok(prompt, return_tensors="pt").to(lm.device)
        gen = lm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)   # greedy = reproducible
        out.append(tok.decode(gen[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return out

def fill_missing_with_mt(pairs_path):
    """Fill empty en_text cells with machine translation and mark them 'machine'. Edits the file in place."""
    df = pd.read_csv(pairs_path, encoding="utf-8-sig")
    empty = df["en_text"].isna() | (df["en_text"].astype(str).str.strip() == "")
    if not empty.any():
        print("nothing to translate"); return df
    df.loc[empty, "en_text"] = machine_translate(df.loc[empty, "ar_text"].tolist())
    df.loc[empty, "translation"] = "machine"
    df.to_csv(pairs_path, index=False, encoding="utf-8-sig")
    print(f"machine-translated {int(empty.sum())} rows")
    return df

# fill_missing_with_mt(PAIRS_CSV)     # <- uncomment only if you decide to use it

## 3. Validate the pairs

Cheap checks that catch typos before they silently corrupt the corpus.

In [ ]:
def check_pairs(passages, pairs_path):
    """Return a list of problems with the pairs file (empty list = fine)."""
    pairs = pd.read_csv(pairs_path, encoding="utf-8-sig")
    ids = {p["id"] for p in passages if p["language"] == "ar"}
    have = {p["id"] for p in passages}
    problems = [f"{twin_id(a)}: already exists" for a in pairs["ar_id"] if twin_id(a) in have]
    for _, r in pairs.iterrows():
        if r["ar_id"] not in ids:                         problems.append(f"{r['ar_id']}: not an Arabic passage id")
        if pd.isna(r["en_text"]) or not str(r["en_text"]).strip(): problems.append(f"{r['ar_id']}: empty en_text")
    dup = pairs["ar_id"][pairs["ar_id"].duplicated()].tolist()
    problems += [f"{d}: listed twice" for d in dup]
    return problems

problems = check_pairs(passages, PAIRS_CSV)
print("\n".join(problems) if problems else "pairs file OK")

## 4. Build the new corpus

* `add_english_pairs` appends one English row per Arabic one. The twin gets id `FEMA-Gxx-en`, language `en`, the English source line, and both
  rows share `pair_id` (as the NCEMA and READY passages already do).
* `add_embed_text` builds `embed_text = "<source>: <passage>"`. Only this is embedded, the shown text is unchanged.
  The prefix gives a one-sentence snippet the context (who said it) that a dense model otherwise lacks.
  Notebook 07 tests twins **with** and **without** the prefix, so you can see which part helps.

In [ ]:
def add_english_pairs(passages, pairs_path, en_source):
    """Append English twins of Arabic passages (see pairs file). Returns a new list.
    The Arabic row gets pair_id too, and the twin gets the English source line."""
    pairs = pd.read_csv(pairs_path, encoding="utf-8-sig")
    twins = {r["ar_id"]: r for _, r in pairs.iterrows()}
    out = []
    for p in passages:
        if p["id"] in twins:
            p = {**p, "pair_id": p["id"][:-3]}
        out.append(p)
    for p in list(out):
        if p["id"] in twins:
            r = twins[p["id"]]
            out.append({**p, "id": twin_id(p["id"]), "language": "en", "source": en_source,
                        "text": str(r["en_text"]).strip(), "translation": r.get("translation", "human")})
    return out

def add_embed_text(passages):
    """Add embed_text = '<source>: <text>' (embedding only; the shown text is unchanged)."""
    return [{**p, "embed_text": f"{p['source']}: {p['text']}"} for p in passages]

new_passages = add_embed_text(add_english_pairs(passages, PAIRS_CSV, EN_SOURCE))
print(len(passages), "->", len(new_passages), "passages")
print(new_passages[-1]["embed_text"][:200])

## 5. Check that each pair is really a translation of the other

bge-m3 is multilingual, so an Arabic passage and its English twin should be very close. A low similarity usually means
the wrong English sentence was pasted. Inspect the lowest rows.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-m3")

def pair_alignment(model, passages):
    """Cosine similarity between each Arabic passage and its English twin, lowest first."""
    by_id = {p["id"]: p for p in passages}
    twins = [p for p in passages if p["id"].startswith(FEMA_ID_PREFIX) and p["language"] == "en"]
    ars = [by_id[t["id"][:-3] + "-ar"] for t in twins]
    ar = model.encode([a["text"] for a in ars], normalize_embeddings=True)
    en = model.encode([t["text"] for t in twins], normalize_embeddings=True)
    sim = (ar * en).sum(axis=1)
    return (pd.DataFrame({"ar_id": [a["id"] for a in ars], "similarity": sim,
                          "english": [t["text"][:90] for t in twins]})
              .sort_values("similarity").reset_index(drop=True))

def plot_alignment(df, path=None):
    fig, ax = plt.subplots(figsize=(6.5, 3))
    ax.hist(df["similarity"], bins=20, color="#0A62A3")
    ax.set_xlabel("Arabic vs English twin, cosine similarity"); ax.set_ylabel("pairs")
    fig.tight_layout()
    if path: fig.savefig(path, dpi=200)
    return fig

align = pair_alignment(model, new_passages)
display(align.head(8))      # inspect these by hand
plot_alignment(align);

## 6. Save the new corpus

In [ ]:
pd.DataFrame(new_passages).to_csv(f"{DRIVE_DIR}/passages_v3.csv", index=False, encoding="utf-8-sig")
print("saved passages_v3.csv")

## 7. Fresh evaluation queries

Written **before** looking at any new retrieval output. Edit freely, but do not edit after you have seen results.
Three groups: English questions, Arabic questions (ask a fluent reader to check the wording), and three out-of-scope questions
whose correct answer is "no guidance found" (`answerable: False`), which test the abstention behaviour.

In [ ]:
fresh_queries = [
 # English, answerable
 {"qid": "N01", "text": "How do I know if my house is structurally damaged after floodwater?", "language": "en"},
 {"qid": "N02", "text": "Should I take photos of the damage before cleaning up?", "language": "en"},
 {"qid": "N03", "text": "Can I use electrical appliances that got wet in the flood?", "language": "en"},
 {"qid": "N04", "text": "Where can I safely run a generator after a flood?", "language": "en"},
 {"qid": "N05", "text": "What should I do if a power line is lying in the floodwater?", "language": "en"},
 {"qid": "N06", "text": "Is it safe to drive through a flooded road?", "language": "en"},
 {"qid": "N07", "text": "Should I get someone to inspect my building before moving back in?", "language": "en"},
 {"qid": "N08", "text": "How do I stay safe near a wadi that is flowing?", "language": "en"},
 {"qid": "N09", "text": "What should I do with food that touched floodwater?", "language": "en"},
 {"qid": "N10", "text": "Who do I report damaged roads or buildings to?", "language": "en"},
 {"qid": "N11", "text": "Is it safe for children to play in floodwater?", "language": "en"},
 {"qid": "N12", "text": "What should I pack if I have to evacuate?", "language": "en"},
 {"qid": "N13", "text": "How long should I stay away from a flooded area?", "language": "en"},
 # Arabic (needs a fluent reader's check)
 {"qid": "N14", "text": "هل من الآمن العودة إلى المنزل بعد الفيضان؟", "language": "ar"},
 {"qid": "N15", "text": "ماذا أفعل إذا سقطت أسلاك الكهرباء في مياه الفيضان؟", "language": "ar"},
 {"qid": "N16", "text": "هل مياه الصنبور آمنة للشرب بعد الفيضان؟", "language": "ar"},
 {"qid": "N17", "text": "كيف أوثق أضرار المنزل قبل التنظيف؟", "language": "ar"},
 # out of scope: the right behaviour is to return nothing
 {"qid": "N18", "text": "How much does flood insurance cost?", "language": "en", "answerable": False},
 {"qid": "N19", "text": "How do I rebuild a flood-damaged roof?", "language": "en", "answerable": False},
 {"qid": "N20", "text": "How do I claim compensation for flood losses?", "language": "en", "answerable": False},
]

Check the fresh queries are not near-copies of the old ones (token overlap), then save them.

In [ ]:
def tokens(text):
    return set(re.findall(r"\w+", str(text).lower()))

def overlap_with_old(fresh, old_texts, threshold=0.5):
    """Fresh queries whose word overlap (Jaccard) with any old query exceeds the threshold."""
    flagged = []
    for q in fresh:
        a = tokens(q["text"])
        for o in old_texts:
            b = tokens(o)
            j = len(a & b) / max(1, len(a | b))
            if j >= threshold: flagged.append((q["qid"], round(j, 2), o))
    return flagged

old_sheet = pd.read_csv(OLD_SHEET, encoding="utf-8-sig")
old_texts = old_sheet["query"].drop_duplicates().tolist()      # rename "query" if your column differs
print(overlap_with_old(fresh_queries, old_texts) or "no near-duplicates of the old queries")

with open(f"{DRIVE_DIR}/fresh_queries.json", "w", encoding="utf-8") as f:
    json.dump(fresh_queries, f, ensure_ascii=False, indent=1)
print("saved fresh_queries.json")

**Next:** run notebook `07_rag_retrieval_comparison.ipynb`.